In [ ]:
# ------------------------------------------------------------
# Description : 이미지 색상 추출 및 3D 변환 노트북
#   - 1) 사용자가 넣은 강아지 사진을 TripoSR 모델로 3D 모델로 변환
#       - Kaggle에서 확인할 수 있게 k3d 뷰어와 mp4/gif 회전 결과 제공
#   - 2) 같은 이미지에서 배경을 제거하고 전경 색상 비중 추출
#       - 블랙 / 화이트 / 브라운 / 그레이 중 비중이 높은 순서대로 출력
#   - 3) 추출된 색상 순위를 컬러칩으로 시각화
# Date : 2026-05-07
# Author : 지현
# ------------------------------------------------------------

# 3D 변환  + 이미지 색상 추출

강아지 사진 한 장을 넣으면 먼저 3D 모델로 변환하고, 같은 사진에서 색상 비중을 뽑아 순위로 정리    

색상 카테고리를 `블랙 / 화이트 / 브라운 / 그레이` 네 가지로 고정    
픽셀을 네 색상 중 하나에 배정한 뒤, 비중이 높은 순서대로 1순위 순서

구현 모델 및 설명 : 3D 변환은 TripoSR, 색상 추출은 배경 제거 후 전경 픽셀을 분석하는 방식   

## 1. 환경 확인

테스트 시 3D 변환은 GPU가 있으면 훨씬 빨라서 Kaggle에서는 Accelerator를 GPU로 켜고 돌리는 걸 추천.

In [ ]:
import sys
from pathlib import Path

import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device:", device)

## 2. 필요한 패키지 설치 및 안내

처음 실행할 때는 모델 코드와 패키지를 내려받아야 해서 시간이 조금 걸리고,     
중간에 dependency 경고가 떠도 셀이 멈추지 않고 다음으로 넘어가면 일단 진행해도 괜찮은 것으로 확인 했습니다.  

Pillow 관련 ImportError가 난 적이 있어서, Pillow는 한 번 깨끗하게 재설치하도록 넣어뒀습니다.    
설치 후 세션 재시작 안내가 뜨면 재시작하고 1번 셀부터 다시 실행하면 됩니다. 

In [ ]:
# 3D 변환, 이미지 캡션, 배경 제거, 노트북 시각화에 필요한 패키지
!pip install -q --upgrade setuptools
!pip install -q --force-reinstall --no-cache-dir "Pillow>=12.1,<13"
!pip install -q git+https://github.com/tatsy/torchmcubes.git
!pip install -q omegaconf einops trimesh rembg imageio[ffmpeg] gradio xatlas moderngl onnxruntime plotly
!pip install -q --upgrade "huggingface-hub>=0.34,<2" "transformers>=4.41,<5" "websockets>=15.0.1,<16"
!pip install -q --force-reinstall --no-cache-dir "numpy==2.1.3" "scipy==1.15.3"

from pathlib import Path
import numpy as np
import scipy
import PIL
print("NumPy version:", np.__version__)
print("SciPy version:", scipy.__version__)
print("Pillow version:", PIL.__version__)

repo_dir = Path("TripoSR")
if not repo_dir.exists():
    !git clone https://github.com/VAST-AI-Research/TripoSR.git
else:
    print("TripoSR repository already exists:", repo_dir.resolve())

## 3. 사진 선택

`IMAGE_PATH`에 직접 이미지 경로를 넣어주시면 됩니다.<<추천  
비워두면 `Data/`, `/kaggle/input/`, 현재 폴더에서 이미지 파일을 자동으로 찾음.  

3D 결과는 강아지 몸 전체가 잘 보이고, 배경이 단순하고, 발이 잘리지 않은 사진일수록 잘 나옵니다.     
( 추후 앱에 모델 넣고 화면 구현시에도 안내 문구로 넣어둘 예정 ) 

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# 직접 지정 예시:
# IMAGE_PATH = "/kaggle/input/my-dog-image/dog.jpg"
# IMAGE_PATH = "../Data/dog.jpg"
IMAGE_PATH = None

search_roots = [Path("../Data"), Path("Data"), Path("/kaggle/input"), Path(".")]
image_exts = {".jpg", ".jpeg", ".png", ".webp"}

if IMAGE_PATH is None:
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend([p for p in root.rglob("*") if p.suffix.lower() in image_exts])
    if not candidates:
        raise FileNotFoundError("이미지를 찾지 못했습니다. IMAGE_PATH에 사진 경로를 직접 넣어 주세요.")
    image_file = candidates[0]
else:
    image_file = Path(IMAGE_PATH)

image = Image.open(image_file).convert("RGB")

print("사용할 이미지:", image_file.resolve())
print("이미지 크기:", image.size)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.show()
# 여기서 kaggle에 넣어둔 모델 이미지가 떠야 맞게 들어간 거라서 이미지 확인 후 넘어가시면 됩니다 !

## 4. 사진을 3D 모델로 변환

여기서 TripoSR를 실행해서 `.glb` 3D 모델과 회전 영상 파일을 만듬.   
GPU 메모리가 부족하면 `MC_RESOLUTION`을 192나 128로 낮춰서 다시 돌리면 3D모델 잘 나오는 거 확인 해뒀습니다.  
(-> 대신 3D 구현 퀄리티가 낮아질 수 있습니다~!)

In [ ]:
import subprocess

OUTPUT_DIR = Path("output_image_color_3d")
MC_RESOLUTION = 256
CHUNK_SIZE = 8192 if torch.cuda.is_available() else 4096

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(repo_dir / "run.py"),
    str(image_file),
    "--device", device,
    "--output-dir", str(OUTPUT_DIR),
    "--model-save-format", "glb",
    "--render",
    "--foreground-ratio", "0.85",
    "--mc-resolution", str(MC_RESOLUTION),
    "--chunk-size", str(CHUNK_SIZE),
]

print("실행 명령:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

mesh_path = OUTPUT_DIR / "0" / "mesh.glb"
video_path = OUTPUT_DIR / "0" / "render.mp4"

if not mesh_path.exists():
    raise FileNotFoundError(f"3D mesh 파일을 찾지 못했습니다: {mesh_path}")

print("3D 모델:", mesh_path.resolve())
print("회전 영상:", video_path.resolve() if video_path.exists() else "생성되지 않음")

## 5. 노트북에서 3D 모델 확인

Kaggle에서 `<model-viewer>` 방식이 빈 화면으로 뜨는 경우가 있어서, 여기서는 `k3d`로 mesh를 직접 보여줌. 
마우스로 회전 및 확대가능.

발이 떠 보이지 않도록 mesh의 가장 낮은 지점을 바닥 높이로 맞춰서 표시.

In [ ]:
import numpy as np
import trimesh
import plotly.graph_objects as go

loaded = trimesh.load(mesh_path, force="scene")

if isinstance(loaded, trimesh.Scene):
    geometries = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh)]
else:
    geometries = [loaded]

if not geometries:
    raise ValueError("GLB 파일 안에서 표시할 mesh를 찾지 못했습니다.")

combined_mesh = trimesh.util.concatenate(geometries)
vertices = combined_mesh.vertices.astype(np.float32)
faces = combined_mesh.faces.astype(np.uint32)

# 가운데 정렬 + 바닥 맞춤. 발을 바닥에 붙이려고 해둠.
vertices[:, 0] -= vertices[:, 0].mean()
vertices[:, 1] -= vertices[:, 1].mean()
vertices[:, 2] -= vertices[:, 2].min()

scale = np.max(np.linalg.norm(vertices, axis=1))
if scale > 0:
    vertices = vertices / scale

print("vertices:", vertices.shape)
print("faces:", faces.shape)

# Plotly가 너무 많은 삼각형을 한 번에 그리면 Kaggle 브라우저가 느려질 수 있음.
# 그래서 보기용으로만 face를 일부 줄이고, 실제 glb 파일은 원본 그대로 둠.
MAX_DISPLAY_FACES = 80000
display_faces = faces
if len(faces) > MAX_DISPLAY_FACES:
    rng = np.random.default_rng(42)
    selected = rng.choice(len(faces), size=MAX_DISPLAY_FACES, replace=False)
    display_faces = faces[selected]
    print(f"표시용 face 수를 {len(faces)}개에서 {len(display_faces)}개로 줄였습니다.")

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=display_faces[:, 0],
            j=display_faces[:, 1],
            k=display_faces[:, 2],
            color="#b8b8b8",
            opacity=1.0,
            flatshading=False,
            lighting=dict(ambient=0.45, diffuse=0.75, specular=0.25, roughness=0.7),
            lightposition=dict(x=80, y=120, z=180),
        )
    ]
)

fig.update_layout(
    height=620,
    margin=dict(l=0, r=0, t=30, b=0),
    title="3D 모델 미리보기",
    scene=dict(
        aspectmode="data",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.1)),
    ),
)

fig.show()

print("3D 화면이 안 보이면 아래 mp4/gif 셀에서 회전 결과를 확인하면 됩니다.")

## 6. 회전 영상과 gif 확인

3D 위젯이 브라우저나 Kaggle 환경 때문에 안 보일 때도, 영상 파일은 비교적 안정적으로 확인됨.

In [ ]:
from IPython.display import Video, Image as IPyImage
import imageio.v2 as imageio

if video_path.exists():
    display(Video(str(video_path), embed=True, html_attributes="controls loop autoplay muted"))
else:
    print("mp4 파일이 없습니다. 3D 변환 셀에서 --render 옵션이 들어갔는지 확인하세요.")

render_frames = sorted((OUTPUT_DIR / "0").glob("render_*.png"))
gif_path = OUTPUT_DIR / "0" / "rotation.gif"

if render_frames:
    frames = [imageio.imread(frame) for frame in render_frames]
    imageio.mimsave(gif_path, frames, fps=12)
    print("gif 파일:", gif_path.resolve())
    display(IPyImage(filename=str(gif_path)))
else:
    print("gif로 만들 렌더링 프레임이 없습니다.")

## 7. BLIP로 이미지 설명 확인

색상 순위 자체는 픽셀 비중으로 계산.    
BLIP 캡션은 다른분들이 확인하실때 이미지를 빠르게 확인할 수 있게 참고 정보로 같이 남겨두겠습니다.

In [ ]:
# BLIP는 이미지 설명 참고용이라, 여기서 실패해도 3D 변환/색상 추출은 계속 진행되게 처리했습니다.
# Kaggle에서 PIL/Pillow가 섞이면 transformers import 단계에서 멈추는 경우가 있어서 예외 처리로 감싸둠.
caption = "BLIP 이미지 설명 생략"

try:
    from transformers import BlipProcessor, BlipForConditionalGeneration

    blip_model_name = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(blip_model_name)

    if torch.cuda.is_available():
        blip_model = BlipForConditionalGeneration.from_pretrained(
            blip_model_name,
            torch_dtype=torch.float16,
        ).to(device)
    else:
        blip_model = BlipForConditionalGeneration.from_pretrained(blip_model_name).to(device)

    inputs = processor(image, return_tensors="pt").to(device)
    if torch.cuda.is_available():
        inputs = {key: value.to(torch.float16) if value.dtype == torch.float32 else value for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = blip_model.generate(**inputs, max_new_tokens=40)

    caption = processor.decode(output_ids[0], skip_special_tokens=True)
    print("이미지 설명:", caption)
except Exception as exc:
    print("BLIP 이미지 설명은 이번 실행에서 생략합니다.")
    print("사유:", type(exc).__name__, exc)
    print("색상 순위는 아래 픽셀 분석 셀에서 그대로 계산됩니다.")

## 8. 사진에서 색상 비중 계산

배경이 섞이면 색상이 이상하게 잡힐 수 있어서 먼저 배경 제거를 시도, 그 다음 전경 픽셀을 `블랙 / 화이트 / 브라운 / 그레이` 네 가지 중 가장 가까운 색으로 배정.

최종 결과는 비중이 높은 순서대로 정렬.

In [ ]:
import pandas as pd

COLOR_INFO = {
    "블랙": {"rgb": np.array([28, 28, 28]), "hex": "#1c1c1c"},
    "화이트": {"rgb": np.array([238, 235, 226]), "hex": "#eeece2"},
    "브라운": {"rgb": np.array([132, 82, 45]), "hex": "#84522d"},
    "그레이": {"rgb": np.array([130, 130, 130]), "hex": "#828282"},
}

def resize_for_analysis(pil_image, max_side=900):
    image_copy = pil_image.copy()
    image_copy.thumbnail((max_side, max_side))
    return image_copy

def get_center_pixels(pil_image, crop_ratio=0.78):
    # rembg가 Kaggle 패키지 충돌로 안 될 때 쓰는 대체 방식.
    # 강아지가 보통 사진 중앙에 있다고 보고 중앙 영역만 잘라서 배경 영향을 줄임.
    width, height = pil_image.size
    crop_w = int(width * crop_ratio)
    crop_h = int(height * crop_ratio)
    left = max((width - crop_w) // 2, 0)
    top = max((height - crop_h) // 2, 0)
    cropped = pil_image.crop((left, top, left + crop_w, top + crop_h)).convert("RGBA")
    arr = np.array(cropped)
    return arr[:, :, :3].reshape(-1, 3), cropped

def find_triposr_processed_image(output_dir):
    # TripoSR가 3D 만들 때 저장해둔 처리 이미지가 있으면 그걸 색상 분석에 재사용함.
    # 파일명은 실행 환경/버전에 따라 달라질 수 있어서 출력 폴더 안의 이미지 파일을 넓게 확인합니다.
    search_dir = Path(output_dir) / "0"
    if not search_dir.exists():
        return None

    image_files = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
        image_files.extend(search_dir.glob(ext))

    if not image_files:
        return None

    preferred_words = ["rgba", "remove", "foreground", "processed", "input", "image"]
    scored = []
    for file_path in image_files:
        name = file_path.name.lower()
        if name.startswith("render") or "rotation" in name:
            continue
        score = 0
        for idx, word in enumerate(preferred_words):
            if word in name:
                score += 20 - idx

        try:
            candidate = Image.open(file_path).convert("RGBA")
            alpha = np.array(candidate)[:, :, 3]
            transparent_ratio = float((alpha < 250).mean())
            if transparent_ratio > 0.01:
                score += 100
            scored.append((score, file_path))
        except Exception:
            pass

    if not scored:
        return None

    scored.sort(key=lambda item: item[0], reverse=True)
    best_score, best_path = scored[0]
    if best_score <= 0:
        return None

    return best_path

def remove_solid_background_by_edge(rgba_image, tolerance=34):
    # TripoSR 처리 이미지가 회색 배경으로 저장되는 경우가 있음.
    # 이미지 테두리는 대부분 배경이라고 보고, 테두리 대표색과 가까운 픽셀을 분석에서 제외함.
    arr = np.array(rgba_image.convert("RGBA"))
    rgb = arr[:, :, :3].astype(np.float32)
    alpha = arr[:, :, 3]

    edge_pixels = np.concatenate([
        rgb[0, :, :],
        rgb[-1, :, :],
        rgb[:, 0, :],
        rgb[:, -1, :],
    ], axis=0)
    edge_color = np.median(edge_pixels, axis=0)
    color_distance = np.linalg.norm(rgb - edge_color, axis=2)

    foreground_mask = (alpha > 20) & (color_distance > tolerance)
    pixels = arr[:, :, :3][foreground_mask]

    preview = arr.copy()
    preview[~foreground_mask, 3] = 0
    return pixels, Image.fromarray(preview, mode="RGBA"), edge_color

def get_pixels_from_rgba(pil_image):
    rgba = resize_for_analysis(pil_image.convert("RGBA"))
    arr = np.array(rgba)
    alpha = arr[:, :, 3]
    mask = alpha > 20
    pixels = arr[:, :, :3][mask]

    # alpha가 거의 전부 불투명하면 배경이 회색/단색으로 채워졌을 가능성이 높음.
    # 이 경우 테두리 색을 배경으로 보고 한 번 더 제외함.
    opaque_ratio = float((alpha > 250).mean())
    if opaque_ratio > 0.98:
        bg_removed_pixels, bg_removed_preview, edge_color = remove_solid_background_by_edge(rgba)
        if len(bg_removed_pixels) > 500:
            print("단색 배경을 제외하고 색상을 계산합니다. 배경 추정 RGB:", np.round(edge_color).astype(int).tolist())
            return bg_removed_pixels, bg_removed_preview

    if len(pixels) > 500:
        return pixels, rgba
    return None, rgba

def get_foreground_pixels(pil_image):
    # 3D 변환에서 이미 만들어진 처리 이미지가 있으면 먼저 그걸 사용함.
    processed_image_path = find_triposr_processed_image(OUTPUT_DIR)
    if processed_image_path is not None:
        try:
            processed_image = Image.open(processed_image_path).convert("RGBA")
            pixels, preview = get_pixels_from_rgba(processed_image)
            if pixels is not None:
                print("TripoSR 처리 이미지로 색상을 계산합니다:", processed_image_path)
                return pixels, preview
        except Exception as exc:
            print("TripoSR 처리 이미지를 읽지 못해서 다른 방식으로 진행합니다:", type(exc).__name__, exc)

    # 저장된 처리 이미지가 없으면 rembg를 시도하고, 실패하면 중앙 영역 기준으로 색상을 계산합니다.
    small_image = resize_for_analysis(pil_image)
    try:
        from rembg import remove

        rgba = remove(small_image).convert("RGBA")
        arr = np.array(rgba)
        alpha = arr[:, :, 3]
        mask = alpha > 20
        pixels = arr[:, :, :3][mask]
        if len(pixels) > 500:
            return pixels, rgba
    except Exception as exc:
        print("배경 제거 실패. 중앙 영역 기준으로 색상을 계산합니다:", type(exc).__name__, exc)

    return get_center_pixels(small_image)

def assign_color_labels(pixels):
    pixels = pixels.astype(np.float32)
    r, g, b = pixels[:, 0], pixels[:, 1], pixels[:, 2]
    brightness = pixels.mean(axis=1)
    saturation = pixels.max(axis=1) - pixels.min(axis=1)

    labels = np.full(len(pixels), "", dtype=object)

    black_mask = brightness < 70
    white_mask = (brightness > 190) & (saturation < 65)
    gray_mask = (saturation < 38) & (brightness >= 70) & (brightness <= 190)
    brown_mask = (r > 65) & (r > g * 1.05) & (g > b * 1.05) & (brightness >= 45) & (brightness <= 205)

    labels[black_mask] = "블랙"
    labels[white_mask & (labels == "")] = "화이트"
    labels[brown_mask & (labels == "")] = "브라운"
    labels[gray_mask & (labels == "")] = "그레이"

    # 조건에 딱 안 들어간 픽셀은 네 색상 대표값 중 가장 가까운 쪽으로 배정함.
    unknown_mask = labels == ""
    if unknown_mask.any():
        unknown_pixels = pixels[unknown_mask]
        names = list(COLOR_INFO.keys())
        prototypes = np.stack([COLOR_INFO[name]["rgb"] for name in names]).astype(np.float32)
        distances = ((unknown_pixels[:, None, :] - prototypes[None, :, :]) ** 2).sum(axis=2)
        nearest = distances.argmin(axis=1)
        labels[unknown_mask] = np.array(names, dtype=object)[nearest]

    return labels

foreground_pixels, foreground_preview = get_foreground_pixels(image)
labels = assign_color_labels(foreground_pixels)

total = len(labels)
summary_rows = []
for color_name in COLOR_INFO:
    count = int((labels == color_name).sum())
    summary_rows.append({
        "컬러": color_name,
        "픽셀수": count,
        "비중": count / total if total else 0,
    })

color_rank_df = pd.DataFrame(summary_rows).sort_values("비중", ascending=False).reset_index(drop=True)
color_rank_df.insert(0, "순위", range(1, len(color_rank_df) + 1))
color_rank_df["비중(%)"] = (color_rank_df["비중"] * 100).round(2)

main_color = color_rank_df.loc[0, "컬러"]

print("대표 컬러:", main_color)
display(color_rank_df[["순위", "컬러", "비중(%)", "픽셀수"]])

plt.figure(figsize=(5, 5))
plt.imshow(foreground_preview)
plt.title("색상 분석에 사용한 전경")
plt.axis("off")
plt.show()

## 9. 컬러칩으로 보기

순위대로 컬러칩을 보여줌. 제일 왼쪽이 이미지에서 가장 많이 나온 색상.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.8))

for ax, (_, row) in zip(axes, color_rank_df.iterrows()):
    color_name = row["컬러"]
    rank = int(row["순위"])
    hex_color = COLOR_INFO[color_name]["hex"]

    # PIL/ImageDraw가 Kaggle 환경에서 깨지는 경우가 있어서 matplotlib 사각형으로 컬러칩을 그림.
    ax.set_facecolor(hex_color)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, color=hex_color))
    border_color = "#ffbe00" if rank == 1 else "#a5a5a5"
    border_width = 5 if rank == 1 else 2
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(border_color)
        spine.set_linewidth(border_width)
    ax.set_title(f"{int(row['순위'])}순위 {row['컬러']}\n{row['비중(%)']:.2f}%")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle(f"대표 컬러: {main_color}", fontsize=16)
plt.tight_layout()
plt.show()

## 10. 최종 정리

테스트 확인 용이하게 만들어둔 것으로 이미지, 3D 파일, 색상 순위를 한 번에 보여주게 정리함.

In [ ]:
print("입력 이미지:", image_file.resolve())
print("3D 모델:", mesh_path.resolve())
print("회전 영상:", video_path.resolve() if video_path.exists() else "없음")
print("회전 gif:", gif_path.resolve() if gif_path.exists() else "없음")
print("이미지 설명:", globals().get("caption", "BLIP 이미지 설명 생략"))
print("대표 컬러:", main_color)

for _, row in color_rank_df.iterrows():
    print(f"{int(row['순위'])}순위: {row['컬러']} ({row['비중(%)']:.2f}%)")